# Implementing custom operators

A pre-processing pipeline may need a transformation that isn't available in Torchvision (and is therefore also missing from the DALI Torchvision API). In that case, you can implement a custom operator. Implemented correctly, it can be reused across pipelines just like a built-in one, by adding it to a `Compose` list.

In [ ]:
from abc import ABC, abstractmethod
from typing import Literal

import nvidia.dali as dali
import nvidia.dali.fn as fn
import nvidia.dali.experimental.dynamic as ndd
from nvidia.dali._typing import TensorLike
from nvidia.dali.experimental.dynamic._device import DeviceLike
from nvidia.dali.experimental.torchvision.v2.operator import (
    Operator,
    adjust_input,
    get_HWC_from_layout_dynamic,
    get_HWC_from_layout_pipeline,
)


def _stack_int64(*values):
    """Helper: pack scalar DALI values into a single int64 vector."""
    return fn.stack(*[fn.cast(v, dtype=dali.types.INT64) for v in values])

## Object-oriented operators

An example custom operator is shown below.

In [ ]:
class TrimBorder(Operator):
    preprocess_data = get_HWC_from_layout_pipeline

    def __init__(
        self,
        border_size: int,
        device: str = "cpu",
    ):
        super().__init__(device=device)
        self.border_size = border_size

    def _kernel(self, data_input):
        in_h, in_w, _, tensor = data_input
        top = dali.math.min(self.border_size, in_h)
        left = dali.math.min(self.border_size, in_w)
        height = dali.math.max(0, in_h - 2 * self.border_size)
        width = dali.math.max(0, in_w - 2 * self.border_size)
        return fn.slice(
            tensor,
            _stack_int64(left, top),
            _stack_int64(width, height),
            axis_names="WH",
            out_of_bounds_policy="pad",
            fill_values=0,
            device=self.device,
        )

`TrimBorder` is a subclass of `Operator` and implements the two required methods: the constructor `__init__` and `_kernel`. The latter contains the DALI code that implements the transformation logic. When listed as a step in `Compose`, it becomes part of the pipeline just like any built-in operator.

`Operator` is the base class for all object-oriented transformations. The following code highlights its main design points:

```python
class Operator(ABC):
    """
    Abstract base class for operator specification

    Implement _kernel for algorithm specific processing

    ``arg_rules`` - a sequence of verification rules for algorithm's arguments.
    ``input_rules`` - a sequence of verification rules for algorithm's input data.
    ``preprocess_data`` - a function to preprocess the input data.

    Parameters
    ----------
    device : Literal["cpu", "gpu"], optional, default = "cpu"
        Device to use for the operator. Can be ``"cpu"`` or ``"gpu"``.
    **kwargs
        Additional keyword arguments for the operator.
    """

    arg_rules: tuple[ArgumentValidateRule, ...] = []
    input_rules: tuple[DataValidateRule, ...] = []
    preprocess_data = None

    @classmethod
    def verify_args(cls, **kwargs):
        for rule in cls.arg_rules:
            rule.verify(**kwargs)

    @classmethod
    def verify_data(cls, data_input):
        for rule in cls.input_rules:
            rule.verify(data_input)

    def __init__(self, device: Literal["cpu", "gpu"] = "cpu", **kwargs):
        self.device = device
        type(self).verify_args(**kwargs)

    @abstractmethod
    def _kernel(self, data_input):
        """
        Algorithm's processing
        """
        pass

    ...

    def _invoke(self, data_input):
        """
        Private method is used to execute operator from a Compose pipeline
        Note: Do not call directly
        """
        type(self).verify_data(data_input)

        # Original input is transferred to GPU, before being preprocess_data.
        # The preprocess_data creates an arbitrary tuple
        if self.device == "gpu":
            data_input = data_input.gpu()

        if type(self).preprocess_data:
            data_input = type(self).preprocess_data(data_input)

        output = self._kernel(data_input)

        return output
```

A minimal custom operator only needs to implement `_kernel`, which contains the transformation's logic. If the operator takes configuration parameters, implement `__init__` as well. To validate those parameters, provide `arg_rules` — either existing validators or custom ones deriving from `ArgumentValidateRule`. If the input itself needs validation (for example, restricting the supported number of channels), list the corresponding rules in `input_rules`. Finally, if additional data processing is needed outside of the DALI pipeline, set `preprocess_data` to a processing function. When in doubt, check how the existing operators are implemented.

## Functional operators

The functional API is simpler, with no formal rules for implementing a custom operator. However, when DALI Dynamic Mode is used for processing, operators are expected to consume a DALI `Tensor` or `Batch`. Input data can be adjusted automatically by decorating the processing function with the `@adjust_input` decorator, as shown in the example below. It implements the same `TrimBorder` transformation as the operator above:

In [ ]:
@adjust_input
def trim_border(
    inpt: TensorLike | ndd.Batch,
    border_size: int,
    device: DeviceLike = "cpu",
) -> ndd.Tensor | ndd.Batch:
    in_h, in_w, _ = get_HWC_from_layout_dynamic(inpt)
    top = min(border_size, in_h)
    left = min(border_size, in_w)
    height = max(0, in_h - 2 * border_size)
    width = max(0, in_w - 2 * border_size)
    axes = [-3, -2] if inpt.layout[-3:] == "HWC" else [-2, -1]
    return ndd.slice(
        inpt,
        (top, left),
        (height, width),
        axes=axes,
        out_of_bounds_policy="pad",
        fill_values=0,
        device=device,
    )